# Project Lab II: Marketing Attribution, P&L Variance, Product Funnel

Notebook for **Chapter 17** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch17.html

Tip: change a number, predict the output, then run the cell again.

## 17.1 Project: Marketing Campaign and Attribution Analysis

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(16)
channels = ["Search","Social","Email","Display"]
spend = pd.Series({"Search": 120000, "Social": 90000, "Email": 15000, "Display": 60000}, name="spend")
rows = []
for j in range(1, 2001):
    k = rng.integers(1, 5)
    path = rng.choice(channels, k, p=[.35,.3,.15,.2])
    p = 0.05 + 0.08*(path[-1] in ("Email","Search")) + 0.02*(k>=3)
    conv = rng.random() < p
    for i, c in enumerate(path):
        rows.append((j, i, c, int(conv and i == len(path)-1)))
tp = pd.DataFrame(rows, columns=["journey","step","channel","converted"])
tp.to_csv("touchpoints.csv", index=False)
J = tp.groupby("journey").agg(n=("step","size"), conv=("converted","max"))
print("journeys", len(J), "conversions", J.conv.sum(), "rate", round(J.conv.mean()*100,2))
won = tp[tp.journey.isin(J.index[J.conv==1])].copy()
last = won.sort_values("step").groupby("journey").tail(1).channel.value_counts()
first = won.sort_values("step").groupby("journey").head(1).channel.value_counts()
won["w"] = 1/won.groupby("journey").step.transform("size")
linear = won.groupby("channel").w.sum()
out = pd.DataFrame({"first":first,"last":last,"linear":linear}).fillna(0).round(1).loc[channels]
out["spend"] = spend
for m in ["first","last","linear"]:
    out["CAC_"+m] = (out.spend/out[m]).round(0)
print(out)
print(out[["first","last","linear"]].sum().round(1).to_dict())

journeys 2000 conversions 181 rate 9.05
         first  last  linear   spend  CAC_first  CAC_last  CAC_linear
channel                                                              
Search      65    89    76.3  120000    1846.0    1348.0      1573.0
Social      46    35    45.6   90000    1957.0    2571.0      1974.0
Email       39    33    35.7   15000     385.0     455.0       420.0
Display     31    24    23.4   60000    1935.0    2500.0      2564.0
{'first': 181.0, 'last': 181.0, 'linear': 181.0}


In [ ]:
out["ROAS_linear"] = (out.linear*2500/out.spend).round(2)
print(out[["spend","linear","ROAS_linear"]])

          spend  linear  ROAS_linear
channel                                
Search   120000    76.3         1.59
Social    90000    45.6         1.27
Email     15000    35.7         5.95
Display   60000    23.4         0.98


## 17.2 Project: Finance P&L Variance Dashboard

In [ ]:
import pandas as pd
rows = [("Revenue","Income",1000000,1060000),
        ("Cost of goods","Expense",600000,650000),
        ("Salaries","Expense",150000,148000),
        ("Marketing","Expense",60000,72000),
        ("Rent","Expense",40000,40000)]
d = pd.DataFrame(rows, columns=["line","type","budget","actual"])
d["variance"] = d.actual - d.budget
d["fav"] = d.apply(lambda r: r.variance if r.type=="Income" else -r.variance, axis=1)
d["var_pct"] = (d.fav / d.budget * 100).round(1)
d["flag"] = d.fav.apply(lambda v: "FAV" if v>0 else ("ADV" if v<0 else "-"))
print(d.to_string(index=False))
inc = d[d.type=="Income"][["budget","actual"]].sum(); exp = d[d.type=="Expense"][["budget","actual"]].sum()
np_b, np_a = inc.budget-exp.budget, inc.actual-exp.actual
print("Profit budget", np_b, "actual", np_a, "fav", np_a-np_b, round((np_a-np_b)/np_b*100,1), "%")
print("sum of fav =", d.fav.sum())

         line    type  budget  actual  variance    fav  var_pct flag
      Revenue  Income 1000000 1060000     60000  60000      6.0  FAV
Cost of goods Expense  600000  650000     50000 -50000     -8.3  ADV
     Salaries Expense  150000  148000     -2000   2000      1.3  FAV
    Marketing Expense   60000   72000     12000 -12000    -20.0  ADV
         Rent Expense   40000   40000         0      0      0.0    -
Profit budget 150000 actual 150000 fav 0 0.0 %
sum of fav = 0


In [ ]:
bu, bp, au = 10000, 100, 9600
ap = 1060000/au
vol = (au-bu)*bp; price = (ap-bp)*au
print("volume effect", vol, "price effect", round(price), "total", round(vol+price))

volume effect -40000 price effect 100000 total 60000


## 17.3 Project: Product Analytics: Event Tracking Plan and Funnel Analysis

In [ ]:
import numpy as np, pandas as pd, sqlite3
rng = np.random.default_rng(18)
steps = ["app_open","product_view","add_to_cart","checkout_start","purchase"]
rates = [1, .6, .35, .6, .75]
rows = []
for u in range(1, 5001):
    plat = rng.choice(["android","ios","web"], p=[.5,.3,.2])
    t = pd.Timestamp("2025-04-01") + pd.Timedelta(minutes=int(rng.integers(0, 28*1440)))
    for s, r in zip(steps, rates):
        if rng.random() > r: break
        rows.append((u, s, t.isoformat(sep=" "), plat))
        t += pd.Timedelta(minutes=int(rng.integers(1, 30)))
ev = pd.DataFrame(rows, columns=["user_id","event_name","event_time","platform"])
ev.to_csv("events.csv", index=False)
print(len(ev), "events")
con = sqlite3.connect(":memory:"); ev.to_sql("events", con, index=False)
q = """
SELECT
 COUNT(DISTINCT CASE WHEN event_name='app_open' THEN user_id END) AS s1_open,
 COUNT(DISTINCT CASE WHEN event_name='product_view' THEN user_id END) AS s2_view,
 COUNT(DISTINCT CASE WHEN event_name='add_to_cart' THEN user_id END) AS s3_cart,
 COUNT(DISTINCT CASE WHEN event_name='checkout_start' THEN user_id END) AS s4_checkout,
 COUNT(DISTINCT CASE WHEN event_name='purchase' THEN user_id END) AS s5_purchase
FROM events"""
r = pd.read_sql(q, con).iloc[0]; print(r.to_dict())
f = pd.DataFrame({"users": r.values}, index=steps)
f["step_conv_%"] = (f.users / f.users.shift(1) * 100).round(1)
f["overall_%"] = (f.users / f.users.iloc[0] * 100).round(1)
print(f)

10164 events
{'s1_open': 5000, 's2_view': 3019, 's3_cart': 1037, 's4_checkout': 628, 's5_purchase': 480}
                users  step_conv_%  overall_%
app_open         5000          NaN      100.0
product_view     3019         60.4       60.4
add_to_cart      1037         34.3       20.7
checkout_start    628         60.6       12.6
purchase          480         76.4        9.6


In [ ]:
g = ev.groupby(["platform","event_name"]).user_id.nunique().unstack()[steps]
g["open_to_purchase_%"] = (g.purchase/g.app_open*100).round(1)
print(g[["app_open","purchase","open_to_purchase_%"]])

event_name  app_open  purchase  open_to_purchase_%
platform                                          
android         2541       245                 9.6
ios             1467       139                 9.5
web             992        96                 9.7


In [ ]:
ev = pd.read_csv("events.csv", parse_dates=["event_time"])
plan = {"app_open","product_view","add_to_cart","checkout_start","purchase"}
print("unknown events:", (~ev.event_name.isin(plan)).sum())
print("null user_id:", ev.user_id.isna().sum(), "null platform:", ev.platform.isna().sum())
print("duplicates:", ev.duplicated(["user_id","event_name"]).sum())
p = ev.pivot_table(index="user_id", columns="event_name", values="event_time", aggfunc="min")
print("cart before view:", (p.add_to_cart < p.product_view).sum())
d = (p.purchase - p.app_open).dropna().dt.total_seconds()/60
print("median minutes open to purchase:", round(d.median(),1), "n", len(d))

unknown events: 0
null user_id: 0 null platform: 0
duplicates: 0
cart before view: 0
median minutes open to purchase: 61.0


## Chapter recap

## Practice set

**1. (Easy)** Email has linear credit 35.7 and spend 15,000. What is its CAC?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch17.html